# Reproducible training and evidence

The active experiment trains a binary detector across two NetFlow sources. It is not an independently validated production detector. See docs/MODEL_CARD.md for the failed external experiment and the full limitations.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/network_detector.py').exists())


## Reproduce from the repository root

```bash
python3 scripts/fetch_dataset.py
python3 scripts/fetch_dataset.py --dataset nf-bot-iot
python3 scripts/train_detector.py --include-bot
python3 scripts/write_model_card.py
```

Model and probability policy selection use validation data. Do not change choices based on final test metrics. Future research needs a fresh external or temporal holdout.


In [ ]:
report = json.loads((ROOT / 'models/training_report.json').read_text())
pd.DataFrame(report['validation_comparison']).T


In [ ]:
pd.DataFrame(report['validation_policies']).T[['recall', 'false_positive_rate', 'average_precision', 'brier', 'threshold']]


In [ ]:
pd.DataFrame(report['per_attack_test']).T


## Verify the artifact and score real sample records


In [ ]:
import hashlib
import sys
import joblib
from threadpoolctl import threadpool_limits
sys.path.insert(0, str(ROOT))
from src.network_detector import validate_netflow, score_netflow

artifact = ROOT / 'models/netflow_detector.joblib'
assert hashlib.sha256(artifact.read_bytes()).hexdigest() == report['artifact_sha256']
bundle = joblib.load(artifact)
sample = validate_netflow(pd.read_csv(ROOT / 'data/netflow_sample.csv'))
with threadpool_limits(limits=2):
    predictions = score_netflow(bundle, sample)
predictions


## Explainability research

Read docs/EXPLAINABILITY_RESEARCH_2026.md for original sources on counterfactuals, HyConEx, EBMs and conformal methods. SHAP is optional; it explains the base estimator, not causal attack mechanisms. The experimental prediction sets require empirical coverage checks and do not guarantee coverage under drift.
